Deliverable 2 

Ida Sofie Ramsli & Emilie Marthinsen Kålås

-----------

Task 1

In [ ]:
from pathlib import Path
import re


def read_log_file(file_path: Path) -> tuple[str, str]:
    # Read a log file and return its content and the encoding used

    file_path = Path(file_path)

    try:
        # Try to read the file with utf-8 encoding
        with file_path.open("r", encoding="utf-8") as file:
            content = file.read()

        return content, "utf-8"
    
    except UnicodeDecodeError:
        # If utf-8 fails, try to read the file with latin-1 encoding
        with file_path.open("r", encoding="latin-1") as file:
            content = file.read()

        return content, "latin-1"



def parse_log_content(file_text_content: str) -> list[dict]:
    # Extract valid sales transaction records from the log file content

    # Match the pattern of a valid sales transaction record
    pattern = (
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} "
        r"\d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>[^\]]+)\]\s+"
        r"item=(?P<item>\.+?)\s+"
        r"price=(?P<price>\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )



    transactions = []

    # Process each line in the log file content separately

    for line in file_text_content.splitlines():
        match = re.search(pattern, line)

        # Ignore lines that do not match the transaction record pattern
        if match:
            transaction = match.groupdict()

            # Convert the price from a string to a float
            transaction["price"] = float(transaction["price"])

            transactions.append(transaction)

    return transactions


The code firstly tries to read a given file with UFT-8 encoding. If it can't read the file using UFT-8, it then reads the file using Latin-1 encoding instead.

----------

Task 2

In [ ]:
directory = Path("data")


def find_log_files(directory: Path) -> list[Path]:

    directory = Path(directory)

    # Create a list of all the log files
    log_files = []


    for item in directory.iterdir():
        if item.is_dir():
            # If the item is a directory, search inside it recursively
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log":
            # If the item is a log file, add it to the list
            log_files.append(item)


    return log_files



def find_active_registers(data) -> set[str]:

    # Create an empty set to store the unique register IDs
    active_registers = set()

    if isinstance(data, dict):

        if data.get("active") is True and isinstance(data.get("id"), str):
            # If the register is active, add its ID to the set
            active_registers.add(data["id"])

    # Search through all values in the dictionary recursively to find active registers
        for value in data.values():
            active_registers.update(find_active_registers(value))


    elif isinstance(data, list):
        # Search through all items in the list recursively to find active registers
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers


The first function, `find_log_files`, uses recursion in order to collect all the log files from the directory and return a list of all of them. The second function, `find_active_registers`, uses recursion to search the nested dictionaries and lists in the store configuration. It adds the ID's of all the active registers to a set and returns that set. 

-------------

Task 3

In [ ]:
import numpy as np
import pandas as pd


def remove_price_outliers(prices) -> np.ndarray:
    """
    Finds the mean and sigma in the data, and replaces the outliers value with the median of the normal prices.
    Checks the assumption; when over 3*sigma from the mean, is an outlier.  
    """
    arr = np.array(prices, dtype=float)
    mu = np.mean(arr)
    sigma = np.std(arr)

    is_outlier = np.abs(arr - mu) > 3 * sigma
    median_valid = np.median(arr[~is_outlier])

    cleaned = arr.copy()  
    cleaned[is_outlier] = median_valid
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """ 
    Computes the refund ratio per item. Makes sure with 1e-6 
    that the items with 0 sold, does not crash the program 
    """
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)


In this task we cleaned the data for outliers, because it can give us a schewed result. We used the mean and standard deviation to find them, therafter we replaced them with the median to get a result as correct as possible. 

...
skriv mer her


------

Task 4

In [ ]:
def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """ 
    Makes a Data frame for the data, with the index being the timestamps
    """
    df = pd.DataFrame.from_records(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()
    return df


def hourly_revenue_by_item(df: pd.DataFrame) -> pd.Series:
    """Groups the data by the item and, and resamples the transactions 
    in 1 -hour windows, and the price is calculated by the total price 
    """
    return df.groupby("item")["price"].resample("1h").sum()

def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table that displays the total transaction revenue grouped by 
    item across status categories (Success, Refund, Cancelled)
    """
    pivot = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0.0,
    )
    for col in ("SUCCESS", "REFUND", "CANCELLED"):
        if col not in pivot.columns:
            pivot[col] = 0.0
    return pivot[["SUCCESS", "REFUND", "CANCELLED"]]


def compute_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
   """
   Computing the refund ratio per item by using task 3 function

   """
    counts = (
        df.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
    )
    
    for col in ("SUCCESS", "REFUND"):
        if col not in counts.columns:
            counts[col] = 0

    sales = counts["SUCCESS"].to_numpy(dtype=float)
    refunds = counts["REFUND"].to_numpy(dtype=float)
    ratios = compute_refund_ratio(sales, refunds)

    result = pd.DataFrame({
        "item": counts.index,
        "sales_count": sales.astype(int),
        "refund_count": refunds.astype(int),
        "refund_ratio": ratios,
    }).set_index("item")
    
    return result



In this task we converted the list of transaction to a Data frame. We used the timestaps as an index to sort the transactions. In the result 
...

må svares på med resultater
